In [ ]:
import os
from dotenv import load_dotenv
import requests

load_dotenv()  # loads .env file in current directory

In [ ]:
# Basic configuration for Todoist API
API_TOKEN = os.getenv("TODOIST_API_TOKEN")

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

BASE_URL = "https://api.todoist.com/api/v1"

In [ ]:
at_home = False  # Set this to True if you are at home, False if you are traveling
assert isinstance(at_home, bool), "at_home must be a boolean"

# Adding Projects

In [ ]:
# Project Functions

# Get all projects
def get_projects():
    url = f"{BASE_URL}/projects"
    all_projects = []
    cursor = None

    while True:
        params = {"limit": 50}
        if cursor:
            params["cursor"] = cursor

        response = requests.get(url, headers=HEADERS, params=params)
        response.raise_for_status()
        data = response.json()

        all_projects.extend(data["results"])

        # stop if no more pages
        cursor = data.get("next_cursor")
        if not cursor:
            break

    return all_projects

# Check if a project with the same name already exists
def project_exists(project, existing_projects):
    target_name = project.get("name")
    return any(p.get("name") == target_name for p in existing_projects)

# Create a new project
def create_project(project):
    url = f"{BASE_URL}/projects"
    response = requests.post(url, headers=HEADERS, json=project)
    response.raise_for_status()
    return response.json()

In [ ]:
# Projects to add
projects_to_add = [
    {"name": "Weekly review"}
]

# Fetch existing projects
existing_projects = get_projects()

print(len(existing_projects))
print([p["name"] for p in existing_projects])

In [ ]:
# Create projects if needed
for project in projects_to_add:
    if not project_exists(project, existing_projects):
        response = create_project(project)
        print(response)
    else:
        print(f"Project already exists: {project['name']}")

In [ ]:
# Fetch again
projects = get_projects()

# Build name → id lookup
project_summary = {p["name"]: p["id"] for p in projects}

In [ ]:
# Extract IDs safely
project_WR = project_summary.get("Weekly review")
project_admin = project_summary.get("Admin/Procedural")
project_calls = project_summary.get("Calls")
project_kitchen = project_summary.get("At the kitchen/cooking")
assert project_WR is not None, "Project 'Weekly review' not found"
assert project_admin is not None, "Project 'Admin/Procedural' not found"
assert project_calls is not None, "Project 'Calls' not found"
assert project_kitchen is not None, "Project 'At the kitchen/cooking' not found"

In [ ]:
url = f"{BASE_URL}/projects/{project_WR}"

data = {
    "child_order": 0
}

response = requests.post(url, headers=HEADERS, json=data)

assert response.status_code == 200, response.text

# Adding Sections

In [ ]:
# Section Functions

# Get sections
def get_sections():
    url = f"{BASE_URL}/sections"
    all_sections = []
    cursor = None

    while True:
        params = {"limit": 50}
        if cursor:
            params["cursor"] = cursor

        response = requests.get(url, headers=HEADERS, params=params)
        response.raise_for_status()
        data = response.json()

        all_sections.extend(data["results"])

        cursor = data.get("next_cursor")
        if not cursor:
            break

    return all_sections

# Create a new section
def create_section(section):
    url = f"{BASE_URL}/sections"
    response = requests.post(url, headers=HEADERS, json=section)
    response.raise_for_status()
    return response.json()

# Check if a section with the same name already exists in the project
def section_exists(section, existing_sections):
    return any(
        (s["name"] == section["name"]) and 
        (s["project_id"] == section["project_id"])
        for s in existing_sections
    )

# Get section ID by name
def get_id_by_name(sections, target_name):
    for s in sections:
        if s["name"] == target_name:
            return s["id"]
    return None

In [ ]:
section_to_add = [
    {"name": "Before my weekly review", "project_id": project_WR},
    {"name": "Get Clear", "project_id": project_WR},
    {"name": "Get Current", "project_id": project_WR},
    {"name": "Get Creative", "project_id": project_WR},
]

In [ ]:
existing_sections = get_sections()
existing_sections = [s for s in existing_sections if s["project_id"] == project_WR]

print(existing_sections)

In [ ]:
for section in section_to_add:
    if not section_exists(section, existing_sections):
        response = create_section(section)
        print(response)
    else:
        print(f"Section already exists: {section['name']}")

In [ ]:
existing_sections = get_sections()
sections_WR = [s for s in existing_sections if s["project_id"] == project_WR]

section_summary = {s["name"]: s["id"] for s in sections_WR}

In [ ]:
sec_before_WR = section_summary.get('Before my weekly review')
sec_get_clear = section_summary.get('Get Clear')
sec_get_current = section_summary.get('Get Current')
sec_get_creative = section_summary.get('Get Creative')

assert sec_before_WR is not None, "Missing section: Before my weekly review"
assert sec_get_clear is not None, "Missing section: Get Clear"
assert sec_get_current is not None, "Missing section: Get Current"
assert sec_get_creative is not None, "Missing section: Get Creative"

# Adding Tasks

In [ ]:
# Task Functions

# Check if a task with the same content already exists in the project
def task_exists(task, existing_tasks):
    return any(
        (t["content"] == task["content"]) and 
        (t.get("project_id") == task.get("project_id"))
        for t in existing_tasks
    )

# Get tasks 
def get_tasks(project_id):
    url = f"{BASE_URL}/tasks"
    response = requests.get(url, headers=HEADERS, params={"project_id": project_id})
    response.raise_for_status()
    return response.json()["results"]

# Create new tasks
def create_task(task):
    url = f"{BASE_URL}/tasks"
    response = requests.post(url, headers=HEADERS, json=task)
    response.raise_for_status()
    return response.json()

In [ ]:
tasks_to_add = [
    {
        "content": "Call my parents",
        "project_id": project_calls
    },
    {
        "content": "Make agua de Jamaica",
        "description": "https://patijinich.com/es/agua-de-jamaica/",
        "project_id": project_kitchen
    },

    # Weekly Review

    {
        "content": "Process my photos for last week",
        "project_id": project_WR,
        "section_id": sec_get_clear
    },
    {
        "content": "Process my todoist inbox",
        "project_id": project_WR,
        "section_id": sec_get_clear
    },
    {
        "content": "Process my revekka97 gmail",
        "project_id": project_WR,
        "section_id": sec_get_clear
    },
    {
        "content": "Process my revekka.gershovich gmail",
        "project_id": project_WR,
        "section_id": sec_get_clear
    },
    {
        "content": "Review previous and future week calendar data",
        "project_id": project_WR,
        "section_id": sec_get_current
    },
    {
        "content": "Review my tickler file for next week",
        "project_id": project_WR,
        "section_id": sec_get_current
    },
    {
        "content": "Review all my personal projects",
        "project_id": project_WR,
        "section_id": sec_get_current
    },
    {
        "content": "Review all my academic projects",
        "project_id": project_WR,
        "section_id": sec_get_current
    },
    {
        "content": "Review all my actions lists",
        "project_id": project_WR,
        "section_id": sec_get_current
    },
    {
        "content": "Review my waiting for list",
        "project_id": project_WR,
        "section_id": sec_get_current
    },
    {
        "content": "Review my agendas for Sasha",
        "project_id": project_WR,
        "section_id": sec_get_creative
    }
]

In [ ]:
existing_tasks = []

for pid in {t["project_id"] for t in tasks_to_add}:
    existing_tasks.extend(get_tasks(pid))

In [ ]:
for task in tasks_to_add:
    if not task_exists(task, existing_tasks):
        response = create_task(task)
        print(f"Created: {task['content']}")
    else:
        print(f"Already exists: {task['content']}")

In [ ]:
if at_home:
    tasks_to_add = [
        {
            "content": "Prepare the desk",
            "project_id": project_WR,
            "section_id": sec_before_WR
        },
        {
            "content": "Programar las clases de español para la semana",
            "project_id": project_admin,
            "description": "https://jesusguzman.my.canva.site/",
            "labels": ["Aprender_español"]
        },
        {
            "content": "Process the inbox in the office",
            "project_id": project_WR,
            "section_id": sec_get_clear
        },
        {
            "content": "Process my mail box",
            "project_id": project_WR,
            "section_id": sec_get_clear
        }]

In [ ]:
if at_home:
    existing_tasks = []

    for pid in {t["project_id"] for t in tasks_to_add}:
        existing_tasks.extend(get_tasks(pid))

In [ ]:
if at_home:
    for task in tasks_to_add:
        if not task_exists(task, existing_tasks):
            response = create_task(task)
            print(f"Created: {task['content']}")
        else:
            print(f"Already exists: {task['content']}")